# 08_4 DQA-SBA full warmup-to-phase2 policy

目的は、DQAをPhase 2のpseudo box選別ではなく、FedSTO Phase 1のbackbone適応を制御する仕組みとして入れること。

- warmup: server labeled dataで通常のsupervised初期化
- Phase 1: backbone-only training + DQA-SBA aggregation
- Phase 2: 短めのfull-parameter refinement + 既存tri-stage gate

DQA-SBAでは、pseudo-label statsはheadを直接動かすためではなく、client backbone residualをglobalにどれくらい混ぜるかを決めるために使う。GTでbest checkpointを選ぶ思想は入れない。


In [1]:
from __future__ import annotations

import json
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else start.resolve()
    required = (
        "dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase1_dqa_sba_full_policy.py",
        "dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py",
        "navigating_data_heterogeneity/setup_fedsto_scene_reproduction.py",
    )
    for base in (start, *start.parents):
        for candidate in (base, base / "Object_Detection"):
            if all((candidate / marker).exists() for marker in required):
                return candidate.resolve()
    raise FileNotFoundError("Could not locate /app/Object_Detection")


REPO_ROOT = find_repo_root()
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
RUN_SCRIPT = DQA_ROOT / "run_dqa_cwa_fedsto_scene_v2_phase1_dqa_sba_full_policy.py"
EVAL_SCRIPT = DQA_ROOT / "evaluate_scene_protocol.py"
POLICY_MODEL = DQA_ROOT / "threshold_policy_model" / "artifacts" / "dqa05_threshold_policy.joblib"

WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_4_scene_phase1_dqa_sba_full_policy"
STATS_ROOT = DQA_ROOT / "stats_dqa08_4_scene_phase1_dqa_sba_full_policy"
LOG_ROOT = DQA_ROOT / "logs_dqa08_4_scene_phase1_dqa_sba_full_policy"
RUNNER_LOG = LOG_ROOT / "runner.out"
TRAIN_LOG = LOG_ROOT / "train.log"
PID_PATH = LOG_ROOT / "runner.pid"
THRESHOLD_LOG = STATS_ROOT / "phase1_dqa_sba_full_policy_schedule.jsonl"

preferred_python = Path("/root/micromamba/envs/al_yolov8/bin/python")
PYTHON_BIN = preferred_python if preferred_python.exists() else Path(sys.executable)

for path in [WORK_ROOT, STATS_ROOT, LOG_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("repo_root:", REPO_ROOT)
print("run_script:", RUN_SCRIPT, RUN_SCRIPT.exists())
print("workspace:", WORK_ROOT)
print("stats_root:", STATS_ROOT)
print("python:", PYTHON_BIN)
print("policy_model:", POLICY_MODEL, POLICY_MODEL.exists())
print("threshold_log:", THRESHOLD_LOG)


repo_root: /app/Object_Detection
run_script: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase1_dqa_sba_full_policy.py True
workspace: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_4_scene_phase1_dqa_sba_full_policy
stats_root: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_4_scene_phase1_dqa_sba_full_policy
python: /root/micromamba/envs/al_yolov8/bin/python
policy_model: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/threshold_policy_model/artifacts/dqa05_threshold_policy.joblib True
threshold_log: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_4_scene_phase1_dqa_sba_full_policy/phase1_dqa_sba_full_policy_schedule.jsonl


In [2]:
# Full pipeline schedule.  This is intentionally a first controlled run, not a 100-round paper-scale run.
WARMUP_EPOCHS = 20
PHASE1_ROUNDS = 12
PHASE2_ROUNDS = 12
DQA_START_PHASE = 1

BATCH_SIZE = 160
WORKERS = 8
REQUESTED_GPUS = 2
MIN_FREE_GIB = 8

# Phase 1 DQA-SBA: quality-weighted, server-anchored, decaying backbone residual.
PHASE1_RESIDUAL_START = 0.45
PHASE1_RESIDUAL_END = 0.18
PHASE1_MAX_RELATIVE_UPDATE = 0.04
PHASE1_SERVER_ANCHOR = 1.25
PHASE1_TEMPERATURE = 2.0
PHASE1_STABILITY_LAMBDA = 0.35
PHASE1_UNIFORM_MIX = 0.05

# Phase 2 is deliberately conservative: full-parameter refinement, but small client residual.
CLASSWISE_BLEND = 0.08
SERVER_ANCHOR = 10.0
TEMPERATURE = 2.4
STABILITY_LAMBDA = 0.55

# Tri-stage pseudoGT policy inherited from 08.  It starts adapting after early phase2 warm rounds.
CLIENT_LR0 = 3e-4
SERVER_LR0 = 1e-3
ADAPT_START_ROUND = 3
POLICY_HORIZON_ROUNDS = PHASE2_ROUNDS
LOW_MID_OBJ_WEIGHT = 0.45
MID_HIGH_OBJ_WEIGHT = 1.0

FORCE_WARMUP = False
FORCE_RESTART = False
APPEND_TRAIN_LOG = False
RUN_TRAINING = True
RUN_IN_BACKGROUND = True
STREAM_TRAIN_OUTPUT = False

try:
    import torch
    AVAILABLE_CUDA_GPUS = torch.cuda.device_count()
except Exception as exc:
    AVAILABLE_CUDA_GPUS = 0
    print("Could not inspect CUDA devices:", exc)

GPUS = min(REQUESTED_GPUS, AVAILABLE_CUDA_GPUS) if AVAILABLE_CUDA_GPUS else 1
if GPUS != REQUESTED_GPUS:
    print(f"Requested {REQUESTED_GPUS} GPU(s), visible={AVAILABLE_CUDA_GPUS}; using GPUS={GPUS}")


def find_free_port(preferred: int) -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        try:
            sock.bind(("127.0.0.1", preferred))
            return preferred
        except OSError:
            pass
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return int(sock.getsockname()[1])


MASTER_PORT = find_free_port(30140)

os.environ["DQA08_POLICY_MODEL"] = str(POLICY_MODEL)
os.environ["DQA08_POLICY_HORIZON_ROUNDS"] = str(POLICY_HORIZON_ROUNDS)
os.environ["DQA08_CLIENT_LR0"] = str(CLIENT_LR0)
os.environ["DQA08_SERVER_LR0"] = str(SERVER_LR0)
os.environ["DQA08_ADAPT_START_ROUND"] = str(ADAPT_START_ROUND)
os.environ["DQA08_LOW_MID_OBJ_WEIGHT"] = str(LOW_MID_OBJ_WEIGHT)
os.environ["DQA08_MID_HIGH_OBJ_WEIGHT"] = str(MID_HIGH_OBJ_WEIGHT)
os.environ["DQA08_THRESHOLD_LOG"] = str(THRESHOLD_LOG)

os.environ["DQA084_PHASE1_RESIDUAL_START"] = str(PHASE1_RESIDUAL_START)
os.environ["DQA084_PHASE1_RESIDUAL_END"] = str(PHASE1_RESIDUAL_END)
os.environ["DQA084_PHASE1_MAX_RELATIVE_UPDATE"] = str(PHASE1_MAX_RELATIVE_UPDATE)
os.environ["DQA084_PHASE1_SERVER_ANCHOR"] = str(PHASE1_SERVER_ANCHOR)
os.environ["DQA084_PHASE1_TEMPERATURE"] = str(PHASE1_TEMPERATURE)
os.environ["DQA084_PHASE1_STABILITY_LAMBDA"] = str(PHASE1_STABILITY_LAMBDA)
os.environ["DQA084_PHASE1_UNIFORM_MIX"] = str(PHASE1_UNIFORM_MIX)
os.environ["DQA084_PHASE1_ROUNDS"] = str(PHASE1_ROUNDS)
os.environ.setdefault("ET_SKIP_AFTER_TRAIN_BEST_VAL", "1")

print("MASTER_PORT:", MASTER_PORT)
print("GPUS:", GPUS)


MASTER_PORT: 30140
GPUS: 2


## Dry Run

設定ファイルとデータリスト生成だけを確認する。長い学習はまだ走らない。


In [3]:
base_cmd = [
    str(PYTHON_BIN),
    str(RUN_SCRIPT),
    "--workspace-root", str(WORK_ROOT),
    "--stats-root", str(STATS_ROOT),
    "--warmup-epochs", str(WARMUP_EPOCHS),
    "--phase1-rounds", str(PHASE1_ROUNDS),
    "--phase2-rounds", str(PHASE2_ROUNDS),
    "--dqa-start-phase", str(DQA_START_PHASE),
    "--batch-size", str(BATCH_SIZE),
    "--workers", str(WORKERS),
    "--gpus", str(GPUS),
    "--master-port", str(MASTER_PORT),
    "--min-free-gib", str(MIN_FREE_GIB),
    "--classwise-blend", str(CLASSWISE_BLEND),
    "--server-anchor", str(SERVER_ANCHOR),
    "--temperature", str(TEMPERATURE),
    "--stability-lambda", str(STABILITY_LAMBDA),
    "--localize-bn",
    "--enable-dqa-guard",
    "--dqa-drop-ratio-threshold", "0.15",
    "--dqa-spike-ratio-threshold", "3.0",
]

dry_cmd = [*base_cmd[:2], "--dry-run", *base_cmd[2:]]
print(" ".join(dry_cmd))
subprocess.run(dry_cmd, cwd=REPO_ROOT, check=True, env=os.environ.copy())


/root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase1_dqa_sba_full_policy.py --dry-run --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_4_scene_phase1_dqa_sba_full_policy --stats-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_4_scene_phase1_dqa_sba_full_policy --warmup-epochs 20 --phase1-rounds 12 --phase2-rounds 12 --dqa-start-phase 1 --batch-size 160 --workers 8 --gpus 2 --master-port 30140 --min-free-gib 8 --classwise-blend 0.08 --server-anchor 10.0 --temperature 2.4 --stability-lambda 0.55 --localize-bn --enable-dqa-guard --dqa-drop-ratio-threshold 0.15 --dqa-spike-ratio-threshold 3.0
DQA08_4 profile: phase1 DQA-SBA full warmup-to-phase2
DQA08_4 protocol: dqa08_4_scene_phase1_dqa_sba_full_policy_v1
DQA08_4 phase1 residual schedule: 0.45 -> 0.18
DQA08_4 phase1 max relative update: 0.04
DQA08_4 policy mo

CompletedProcess(args=['/root/micromamba/envs/al_yolov8/bin/python', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase1_dqa_sba_full_policy.py', '--dry-run', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_4_scene_phase1_dqa_sba_full_policy', '--stats-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_4_scene_phase1_dqa_sba_full_policy', '--warmup-epochs', '20', '--phase1-rounds', '12', '--phase2-rounds', '12', '--dqa-start-phase', '1', '--batch-size', '160', '--workers', '8', '--gpus', '2', '--master-port', '30140', '--min-free-gib', '8', '--classwise-blend', '0.08', '--server-anchor', '10.0', '--temperature', '2.4', '--stability-lambda', '0.55', '--localize-bn', '--enable-dqa-guard', '--dqa-drop-ratio-threshold', '0.15', '--dqa-spike-ratio-threshold', '3.0'], returncode=0)

## Train

`RUN_IN_BACKGROUND=True` のときはPIDとlogだけ残してバックグラウンドで走る。途中確認は次のStatusセルを再実行する。


In [4]:
def read_pid(path: Path) -> int | None:
    if not path.exists():
        return None
    try:
        return int(path.read_text(encoding="utf-8").strip())
    except ValueError:
        return None


def pid_state(pid: int | None) -> str:
    if pid is None:
        return "missing"
    result = subprocess.run(["ps", "-o", "stat=", "-p", str(pid)], capture_output=True, text=True)
    state = result.stdout.strip()
    if result.returncode != 0 or not state:
        return "missing"
    if "Z" in state:
        return "zombie"
    return state


train_cmd = [
    *base_cmd,
    "--log-file", str(TRAIN_LOG),
]
if FORCE_WARMUP:
    train_cmd.append("--force-warmup")
if FORCE_RESTART:
    train_cmd.append("--force-restart")
if APPEND_TRAIN_LOG:
    train_cmd.append("--append-train-log")
if STREAM_TRAIN_OUTPUT:
    train_cmd.append("--stream-train-output")

current_pid = read_pid(PID_PATH)
state = pid_state(current_pid)
print("existing pid:", current_pid, state)
print("command:", " ".join(train_cmd))

if RUN_TRAINING and state not in {"missing", "zombie"}:
    print("Training already appears to be running.")
elif RUN_TRAINING and RUN_IN_BACKGROUND:
    RUNNER_LOG.parent.mkdir(parents=True, exist_ok=True)
    log_mode = "ab" if APPEND_TRAIN_LOG else "wb"
    with RUNNER_LOG.open(log_mode) as out:
        process = subprocess.Popen(
            train_cmd,
            cwd=REPO_ROOT,
            stdout=out,
            stderr=subprocess.STDOUT,
            env=os.environ.copy(),
            start_new_session=True,
        )
    PID_PATH.write_text(str(process.pid), encoding="utf-8")
    print("Started PID:", process.pid)
    print("Runner log:", RUNNER_LOG)
    print("Train log:", TRAIN_LOG)
elif RUN_TRAINING:
    RUNNER_LOG.parent.mkdir(parents=True, exist_ok=True)
    with RUNNER_LOG.open("a", encoding="utf-8", buffering=1) as out:
        process = subprocess.Popen(
            train_cmd,
            cwd=REPO_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            env=os.environ.copy(),
        )
        PID_PATH.write_text(str(process.pid), encoding="utf-8")
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            out.write(line)
        raise SystemExit(process.wait())
else:
    print("RUN_TRAINING=False, not starting.")


existing pid: 2140316 zombie
command: /root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase1_dqa_sba_full_policy.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_4_scene_phase1_dqa_sba_full_policy --stats-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_4_scene_phase1_dqa_sba_full_policy --warmup-epochs 20 --phase1-rounds 12 --phase2-rounds 12 --dqa-start-phase 1 --batch-size 160 --workers 8 --gpus 2 --master-port 30140 --min-free-gib 8 --classwise-blend 0.08 --server-anchor 10.0 --temperature 2.4 --stability-lambda 0.55 --localize-bn --enable-dqa-guard --dqa-drop-ratio-threshold 0.15 --dqa-spike-ratio-threshold 3.0 --log-file /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_4_scene_phase1_dqa_sba_full_policy/train.log
Started PID: 2150146
Runner log: /app/Object_Detection/dynam

In [5]:
# 起動通知。完了通知ではない。
try:
    sys.path.insert(0, str(REPO_ROOT))
    from notebook_notify import notify_discord
    result = notify_discord(
        "08_4 DQA-SBA full warmup-to-phase2 runを起動しました",
        title="実行開始",
        context={"workspace": str(WORK_ROOT), "pid": read_pid(PID_PATH), "runner_log": str(RUNNER_LOG)},
        fail_silently=True,
    )
    print(result)
except Exception as exc:
    print("notify skipped:", exc)


DiscordNotifyResult(ok=True, chunks_sent=1, status_codes=(204,), dry_run=False, error=None)


## Status

学習中はこのセルだけ再実行する。Phase 1のDQA-SBA重みは `dqa_cwa_state.json` の `phase1_dqa_sba` に記録される。


In [6]:
pid = read_pid(PID_PATH)
print("pid:", pid, pid_state(pid))
print("workspace exists:", WORK_ROOT.exists())
print("history:", WORK_ROOT / "history.json", (WORK_ROOT / "history.json").exists())

if (WORK_ROOT / "history.json").exists():
    history = json.loads((WORK_ROOT / "history.json").read_text(encoding="utf-8"))
    print("completed rounds:", len(history))
    print(history[-5:])

state_path = WORK_ROOT / "dqa_cwa_state.json"
if state_path.exists():
    state = json.loads(state_path.read_text(encoding="utf-8"))
    phase1_records = state.get("phase1_dqa_sba", [])
    print("phase1_dqa_sba records:", len(phase1_records))
    if phase1_records:
        print(json.dumps(phase1_records[-1], indent=2)[:3000])

for label, path in [("runner", RUNNER_LOG), ("train", TRAIN_LOG)]:
    print("\n---", label, path)
    if path.exists():
        result = subprocess.run(["tail", "-40", str(path)], capture_output=True, text=True)
        print(result.stdout[-5000:])
    else:
        print("missing")


pid: 2150146 Rsl
workspace exists: True
history: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_4_scene_phase1_dqa_sba_full_policy/history.json False

--- runner /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_4_scene_phase1_dqa_sba_full_policy/runner.out


--- train /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_4_scene_phase1_dqa_sba_full_policy/train.log
enabled=self.cuda):
/app/Object_Detection/navigating_data_heterogeneity/vendor/efficientteacher/trainer/trainer.py:439: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(enabled=self.cuda):

     15/19       45G      2005       640    0.0474   0.04795  0.007365     7.956:  45%|████▌     | 14/31 [00:08<00:10,  1.65it/s]
     15/19       45G      2005       640    0.0474   0.04795  0.007365     7.956:  48%|████▊     | 15/31 [00:08<00:08,  1.94i

## Evaluate

学習完了後に実行する。warmup、最終Phase 1、Phase 2最終をscene splitで評価する。


In [7]:
history_path = WORK_ROOT / "history.json"
global_dir = WORK_ROOT / "global_checkpoints"
available_checkpoints = []
if (global_dir / "round000_warmup.pt").exists():
    available_checkpoints.append(global_dir / "round000_warmup.pt")
if history_path.exists():
    history = json.loads(history_path.read_text(encoding="utf-8"))
    for entry in history:
        path = Path(entry.get("global", ""))
        if path.exists():
            available_checkpoints.append(path)

if not available_checkpoints:
    print("No completed checkpoint yet. Training is probably still in warmup/early rounds; run Status again later, then evaluate.")
else:
    print("available checkpoints:", len(available_checkpoints))
    print("latest:", available_checkpoints[-1])
    eval_cmd = [
        str(PYTHON_BIN),
        str(EVAL_SCRIPT),
        "--workspace", str(WORK_ROOT),
        "--batch-size", "16",
        "--device", "0" if GPUS >= 1 else "",
        "--no-plots",
    ]
    print(" ".join(eval_cmd))
    subprocess.run(eval_cmd, cwd=REPO_ROOT, check=True)

    summary = WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv"
    if summary.exists():
        df = pd.read_csv(summary)
        display(df)
        pivot = df.pivot_table(index="checkpoint", columns="split", values=["map50", "map50_95"], aggfunc="first")
        display(pivot)
    else:
        print("missing summary:", summary)


No completed checkpoint yet. Training is probably still in warmup/early rounds; run Status again later, then evaluate.


In [8]:
# 評価完了後の通知。
try:
    sys.path.insert(0, str(REPO_ROOT))
    from notebook_notify import notify_discord
    summary = WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv"
    if summary.exists():
        df = pd.read_csv(summary)
        msg = df.to_string(index=False)
    else:
        msg = "08_4 evaluation cell finished, but summary CSV was not found."
    result = notify_discord(
        msg,
        title="08_4 評価完了",
        context={"workspace": str(WORK_ROOT), "summary": str(summary)},
        fail_silently=True,
    )
    print(result)
except Exception as exc:
    print("notify skipped:", exc)


DiscordNotifyResult(ok=True, chunks_sent=1, status_codes=(204,), dry_run=False, error=None)
